In [29]:
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path
import warnings
from datetime import datetime, date
from typing import Optional, Union, List
import logging

class MLDataInterface:
    """
    Interface class for managing ML training data with static and dynamic components.
    
    Data Structure:
    - Static data: HDF5 file with basin-indexed features
    - Dynamic data: NetCDF files per source with basin_id × date × features
    """
    
    def __init__(self, data_dir: Union[str, Path], overwrite: bool = False):
        """
        Initialize the ML data interface.
        
        Parameters:
        -----------
        data_dir : str or Path
            Directory to store/load data files
        overwrite : bool
            If True, overwrite existing datasets. If False, append to existing or create new.
        """
        self.data_dir = Path(data_dir)
        self.data_dir.mkdir(parents=True, exist_ok=True)
        
        self.static_file = self.data_dir / "static_features.h5"
        self.dynamic_dir = self.data_dir / "dynamic"
        self.dynamic_dir.mkdir(exist_ok=True)
        
        self.overwrite = overwrite
        
        # Setup logging
        logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
        self.logger = logging.getLogger(__name__)
        
        # Chunking strategy for optimal access patterns
        self.chunk_strategy = {
            'basin_id': 1,  # Adjust based on your typical basin access patterns
            'date': 365,      # ~1 year per chunk to match your access pattern
        }
        
        self.logger.info(f"MLDataInterface initialized at {self.data_dir}")
        self._log_existing_data()
    
    def _log_existing_data(self):
        """Log information about existing data files."""
        if self.static_file.exists():
            self.logger.info(f"Found existing static data: {self.static_file}")
        
        dynamic_files = list(self.dynamic_dir.glob("*.nc"))
        if dynamic_files:
            self.logger.info(f"Found {len(dynamic_files)} dynamic data sources:")
            for f in dynamic_files:
                source_name = f.stem
                try:
                    with xr.open_dataset(f, engine='h5netcdf') as ds:
                        n_basins = len(ds.basin_id)
                        date_range = f"{ds.date.min().values} to {ds.date.max().values}"
                        self.logger.info(f"  {source_name}: {n_basins} basins, {date_range}")
                except Exception as e:
                    self.logger.warning(f"  {source_name}: Could not read metadata ({e})")
    
    def set_static_data(self, df: pd.DataFrame):
        """
        Write/overwrite static data.
        
        Parameters:
        -----------
        df : pd.DataFrame
            DataFrame with basin_id as index and feature columns
        """
        if not isinstance(df.index, pd.Index) or df.index.name is None:
            raise ValueError("DataFrame must have a named index (basin_id)")
            
        if self.static_file.exists() and not self.overwrite:
            self.logger.warning(f"Static data exists and overwrite=False. Overwriting anyway.")
        
        # Save with compression
        df.to_hdf(self.static_file, key='static', mode='w', complevel=6, complib='zlib')
        self.logger.info(f"Static data saved: {df.shape[0]} basins, {df.shape[1]} features")
    
    def get_static_data(self, basin_ids: Optional[List[str]] = None) -> pd.DataFrame:
        """
        Get static data.
        
        Parameters:
        -----------
        basin_ids : list of str, optional
            Specific basin IDs to retrieve. If None, return all.
            
        Returns:
        --------
        pd.DataFrame
            Static features DataFrame
        """
        if not self.static_file.exists():
            raise FileNotFoundError(f"No static data found at {self.static_file}")
        
        df = pd.read_hdf(self.static_file, key='static')
        
        if basin_ids is not None:
            missing_basins = set(basin_ids) - set(df.index)
            if missing_basins:
                self.logger.warning(f"Missing basins in static data: {missing_basins}")
            df = df.loc[df.index.isin(basin_ids)]
        
        return df
    
    def _get_dynamic_file_path(self, source_name: str) -> Path:
        """Get file path for a dynamic data source."""
        return self.dynamic_dir / f"{source_name}.nc"
    
    def set_dynamic_data(self, basin: str, source_name: str, data: pd.DataFrame):
        """
        Insert/overwrite dynamic data for a specific basin and source.
        
        Parameters:
        -----------
        basin : str
            Basin ID
        source_name : str
            Data source name (e.g., 'era5', 'swot')
        data : pd.DataFrame
            DataFrame with date index and feature columns
        """
        if not isinstance(data.index, pd.DatetimeIndex):
            raise ValueError("Data must have DatetimeIndex")
        
        if data.empty:
            self.logger.warning(f"Empty data provided for {basin}, {source_name}")
            return
            
        file_path = self._get_dynamic_file_path(source_name)
        
        # Convert DataFrame to xarray format
        data_vars = {}
        for col in data.columns:
            data_vars[col] = (['date'], data[col].values)
        
        new_ds = xr.Dataset(
            data_vars,
            coords={'date': data.index.values}
        )
        
        if file_path.exists():
            self._update_existing_dynamic_data(file_path, basin, new_ds)
        else:
            self._create_new_dynamic_data(file_path, basin, new_ds)
    
    def _update_existing_dynamic_data(self, file_path: Path, basin: str, new_ds: xr.Dataset):
        """Update existing dynamic data file with new basin data."""
        with xr.open_dataset(file_path, engine='h5netcdf') as existing_ds:
            # Check if basin exists (only if basin_id coordinate exists)
            basin_exists = False
            if 'basin_id' in existing_ds.coords and basin in existing_ds.basin_id.values:
                basin_exists = True
            if basin_exists:
                # Check for overlapping dates
                existing_dates = existing_ds.sel(basin_id=basin).date.values
                new_dates = new_ds.date.values
                overlap = set(pd.to_datetime(existing_dates)) & set(pd.to_datetime(new_dates))
                
                if overlap and not self.overwrite:
                    self.logger.warning(
                        f"Data overlap detected for {basin} in {file_path.stem}: "
                        f"{len(overlap)} overlapping dates. Use overwrite=True to replace."
                    )
                    if not self.overwrite:
                        return
            
            # Combine datasets
            # First, ensure all variables exist in both datasets
            all_vars = set(existing_ds.data_vars) | set(new_ds.data_vars)
            
            # Add missing variables to new_ds
            for var in existing_ds.data_vars:
                if var not in new_ds.data_vars:
                    new_ds[var] = (['date'], np.full(len(new_ds.date), np.nan))
            
            # Expand new_ds to have basin dimension
            new_ds = new_ds.expand_dims('basin_id').assign_coords(basin_id=[basin])
            
            # Combine time and basin dimensions
            if 'basin_id' in existing_ds.coords:
                all_dates = pd.Index(existing_ds.date.values).union(pd.Index(new_ds.date.values))
                all_basins = list(existing_ds.basin_id.values)
                if basin not in all_basins:
                    all_basins.append(basin)
            else:
                # This shouldn't happen if file exists, but handle gracefully
                all_dates = pd.Index(new_ds.date.values) 
                all_basins = [basin]
            
            # Create combined dataset
            combined_ds = xr.Dataset(
                coords={
                    'basin_id': all_basins,
                    'date': all_dates
                }
            )
            
            # Add variables
            for var in all_vars:
                # Initialize with NaN
                combined_ds[var] = (['basin_id', 'date'], 
                                   np.full((len(all_basins), len(all_dates)), np.nan))
                
                # Fill existing data (only if basin_id coordinate exists)
                if var in existing_ds.data_vars and 'basin_id' in existing_ds.coords:
                    for existing_basin in existing_ds.basin_id.values:
                        basin_idx = all_basins.index(existing_basin)
                        for i, date_val in enumerate(existing_ds.date.values):
                            try:
                                date_idx = all_dates.get_loc(pd.Timestamp(date_val))
                                combined_ds[var].values[basin_idx, date_idx] = existing_ds[var].sel(
                                    basin_id=existing_basin, date=date_val
                                ).values
                            except KeyError:
                                # Date not found in combined index, skip
                                continue
                
                # Fill new data
                if var in new_ds.data_vars:
                    basin_idx = all_basins.index(basin)
                    for i, date_val in enumerate(new_ds.date.values):
                        try:
                            date_idx = all_dates.get_loc(pd.Timestamp(date_val))
                            combined_ds[var].values[basin_idx, date_idx] = new_ds[var].sel(
                                basin_id=basin, date=date_val
                            ).values
                        except KeyError:
                            # Date not found in combined index, skip
                            continue
        
        # Save combined dataset
        self._save_dynamic_dataset(combined_ds, file_path)
        self.logger.info(f"Updated {file_path.stem}: basin {basin}, {len(new_ds.date)} dates")
    
    def _create_new_dynamic_data(self, file_path: Path, basin: str, new_ds: xr.Dataset):
        """Create new dynamic data file."""
        # Expand to have basin dimension
        new_ds = new_ds.expand_dims('basin_id').assign_coords(basin_id=[basin])
        
        self._save_dynamic_dataset(new_ds, file_path)
        self.logger.info(f"Created {file_path.stem}: basin {basin}, {len(new_ds.date)} dates")
    
    def _save_dynamic_dataset(self, ds: xr.Dataset, file_path: Path):
        """Save dynamic dataset with optimal encoding."""
        encoding = {}
        for var in ds.data_vars:
            encoding[var] = {
                'chunksizes': (self.chunk_strategy['basin_id'], self.chunk_strategy['date']),
                'compression': 'gzip',
                'compression_opts': 6,
                'shuffle': True,
                'fletcher32': True
            }
        
        ds.to_netcdf(file_path, engine='h5netcdf', encoding=encoding)
    
    def get_dynamic_data(self, basin: str, source: str, 
                        start: Optional[Union[str, datetime, date]] = None,
                        end: Optional[Union[str, datetime, date]] = None) -> pd.DataFrame:
        """
        Get dynamic data for a specific basin and source.
        
        Parameters:
        -----------
        basin : str
            Basin ID
        source : str
            Data source name
        start : str, datetime, or date, optional
            Start date for filtering
        end : str, datetime, or date, optional
            End date for filtering
            
        Returns:
        --------
        pd.DataFrame
            Dynamic data with date index
        """
        file_path = self._get_dynamic_file_path(source)
        
        if not file_path.exists():
            raise FileNotFoundError(f"No dynamic data found for source '{source}'")
        
        with xr.open_dataset(file_path, engine='h5netcdf') as ds:
            if basin not in ds.basin_id.values:
                raise ValueError(f"Basin '{basin}' not found in {source} data")
            
            # Select basin
            basin_data = ds.sel(basin_id=basin)
            
            # Apply date filtering
            if start is not None or end is not None:
                basin_data = basin_data.sel(date=slice(start, end))
            
            # Convert to DataFrame
            df = basin_data.to_dataframe().reset_index()
            df = df.set_index('date').drop(columns=['basin_id'])
            
            # Remove all-NaN rows
            df = df.dropna(how='all')
            
            return df
    
    def list_sources(self) -> List[str]:
        """List available dynamic data sources."""
        return [f.stem for f in self.dynamic_dir.glob("*.nc")]
    
    def list_basins(self, source: Optional[str] = None) -> List[str]:
        """
        List available basins.
        
        Parameters:
        -----------
        source : str, optional
            If provided, list basins for specific source. Otherwise, list from static data.
            
        Returns:
        --------
        list of str
            Basin IDs
        """
        if source is not None:
            file_path = self._get_dynamic_file_path(source)
            if not file_path.exists():
                return []
            with xr.open_dataset(file_path, engine='h5netcdf') as ds:
                return ds.basin_id.values.tolist()
        else:
            if self.static_file.exists():
                df = pd.read_hdf(self.static_file, key='static')
                return df.index.tolist()
            else:
                return []
    
    def get_data_summary(self) -> dict:
        """Get summary of all data in the interface."""
        summary = {
            'static': {},
            'dynamic': {}
        }
        
        # Static data summary
        if self.static_file.exists():
            try:
                df = pd.read_hdf(self.static_file, key='static')
                summary['static'] = {
                    'n_basins': len(df),
                    'n_features': len(df.columns),
                    'features': df.columns.tolist(),
                    'file_size_mb': self.static_file.stat().st_size / 1024 / 1024
                }
            except Exception as e:
                summary['static']['error'] = str(e)
        
        # Dynamic data summary
        for source in self.list_sources():
            file_path = self._get_dynamic_file_path(source)
            try:
                with xr.open_dataset(file_path, engine='h5netcdf') as ds:
                    summary['dynamic'][source] = {
                        'n_basins': len(ds.basin_id),
                        'n_dates': len(ds.date),
                        'date_range': f"{ds.date.min().values} to {ds.date.max().values}",
                        'features': list(ds.data_vars.keys()),
                        'file_size_mb': file_path.stat().st_size / 1024 / 1024
                    }
            except Exception as e:
                summary['dynamic'][source] = {'error': str(e)}
        
        return summary
    
    def validate_data_integrity(self) -> dict:
        """Validate data integrity across static and dynamic datasets."""
        issues = {
            'static': [],
            'dynamic': {}
        }
        
        # Get basin IDs from static data
        static_basins = set()
        if self.static_file.exists():
            try:
                df = pd.read_hdf(self.static_file, key='static')
                static_basins = set(df.index)
            except Exception as e:
                issues['static'].append(f"Cannot read static data: {e}")
        
        # Check dynamic data consistency
        for source in self.list_sources():
            issues['dynamic'][source] = []
            try:
                dynamic_basins = set(self.list_basins(source))
                
                # Check for basins in dynamic but not static
                missing_in_static = dynamic_basins - static_basins
                if missing_in_static:
                    issues['dynamic'][source].append(
                        f"Basins in {source} but not in static: {missing_in_static}"
                    )
                
                # Check data completeness
                file_path = self._get_dynamic_file_path(source)
                with xr.open_dataset(file_path, engine='h5netcdf') as ds:
                    for var in ds.data_vars:
                        nan_ratio = ds[var].isnull().sum() / ds[var].size
                        if nan_ratio > 0.99:  # More than 99% NaN
                            issues['dynamic'][source].append(
                                f"Variable {var} is >99% NaN ({nan_ratio:.1%})"
                            )
                            
            except Exception as e:
                issues['dynamic'][source].append(f"Error validating {source}: {e}")
        
        return issues

In [30]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from tqdm import tqdm

def create_sample_data():
    """Create sample data for testing the interface."""
    
    # Sample basin IDs
    basin_ids = [f"basin_{i:04d}" for i in range(100)]
    
    # Sample static data
    static_data = {
        'elevation_mean': np.random.normal(500, 200, len(basin_ids)),
        'area_km2': np.random.lognormal(3, 1, len(basin_ids)),
        'slope_mean': np.random.gamma(2, 0.5, len(basin_ids)),
        'land_cover_forest': np.random.beta(2, 3, len(basin_ids))
    }
    static_df = pd.DataFrame(static_data, index=basin_ids)
    static_df.index.name = 'basin_id'
    
    # Sample date range
    dates = pd.date_range('2020-01-01', '2022-12-31', freq='D')
    
    return basin_ids, static_df, dates

In [31]:
"""Example 1: Basic initialization and data setting."""
print("=== Example 1: Basic Usage ===")

# Create sample data
basin_ids, static_df, dates = create_sample_data()

# Initialize interface
data_interface = MLDataInterface('./test_ml_data', overwrite=True)

# Set static data
data_interface.set_static_data(static_df)
print(f"Static data set: {static_df.shape}")

# Retrieve static data
retrieved_static = data_interface.get_static_data()
print(f"Retrieved static data: {retrieved_static.shape}")
print("Static features:", retrieved_static.columns.tolist())

2025-09-16 20:19:39,556 - INFO - MLDataInterface initialized at test_ml_data
2025-09-16 20:19:39,557 - INFO - Found existing static data: test_ml_data/static_features.h5
2025-09-16 20:19:39,558 - INFO - Found 1 dynamic data sources:
2025-09-16 20:19:39,579 - INFO -   era5: 1 basins, 2020-01-01T00:00:00.000000000 to 2022-12-31T00:00:00.000000000
2025-09-16 20:19:39,593 - INFO - Static data saved: 100 basins, 4 features


=== Example 1: Basic Usage ===
Static data set: (100, 4)
Retrieved static data: (100, 4)
Static features: ['elevation_mean', 'area_km2', 'slope_mean', 'land_cover_forest']


In [32]:
"""Example 2: Adding dynamic data for different sources."""
print("\n=== Example 2: Dynamic Data Insertion ===")

# Add ERA5 climate data for several basins
print("\nAdding ERA5 data...")
for i, basin in tqdm(enumerate(basin_ids[:5])):  # First 5 basins
    # Create sample ERA5 data
    era5_data = {
        'temperature': np.random.normal(15, 10, len(dates)),
        'precipitation': np.random.exponential(2, len(dates)),
        'humidity': np.random.normal(70, 20, len(dates))
    }
    era5_df = pd.DataFrame(era5_data, index=dates)
    
    # Insert data
    data_interface.set_dynamic_data(basin, 'era5', era5_df)


=== Example 2: Dynamic Data Insertion ===

Adding ERA5 data...


0it [00:00, ?it/s]2025-09-16 20:19:42,667 - INFO - Updated era5: basin basin_0000, 1096 dates
1it [00:01,  1.73s/it]2025-09-16 20:19:48,714 - INFO - Updated era5: basin basin_0001, 1096 dates
2it [00:07,  4.27s/it]2025-09-16 20:19:59,507 - INFO - Updated era5: basin basin_0002, 1096 dates
3it [00:18,  7.25s/it]2025-09-16 20:20:15,222 - INFO - Updated era5: basin basin_0003, 1096 dates
4it [00:34, 10.59s/it]2025-09-16 20:20:37,811 - INFO - Updated era5: basin basin_0004, 1096 dates
5it [00:56, 11.38s/it]


In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import h5py
import os
from pathlib import Path
import time

# For testing on smaller scale
N_BASINS = 1000  # Scale down from 100k for testing
N_YEARS = 5      # Scale down from 40 for testing
START_DATE = '2020-01-01'
END_DATE = '2024-12-31'

print(f"Testing with {N_BASINS} basins and {N_YEARS} years")
print(f"Date range: {START_DATE} to {END_DATE}")

Testing with 1000 basins and 5 years
Date range: 2020-01-01 to 2024-12-31


In [2]:
# Generate basin IDs (strings as in your case)
basin_ids = [f"basin_{i:06d}" for i in range(N_BASINS)]

# Create date range
dates = pd.date_range(START_DATE, END_DATE, freq='D')
print(f"Generated {len(dates)} days of data")

# Static features (constant per basin)
np.random.seed(42)
static_features = {
    'elevation_mean': np.random.normal(500, 200, N_BASINS),
    'area_km2': np.random.lognormal(3, 1, N_BASINS),
    'slope_mean': np.random.gamma(2, 0.5, N_BASINS),
    'land_cover_forest': np.random.beta(2, 3, N_BASINS),
}

static_df = pd.DataFrame(static_features, index=basin_ids)
print(f"Static data shape: {static_df.shape}")
print(static_df.head())

Generated 1827 days of data
Static data shape: (1000, 4)
              elevation_mean   area_km2  slope_mean  land_cover_forest
basin_000000      599.342831  81.398385    0.469070           0.543127
basin_000001      472.347140  50.634526    0.743485           0.419354
basin_000002      629.537708  21.319675    0.111240           0.333050
basin_000003      804.605971  10.517739    0.978735           0.074499
basin_000004      453.169325  40.375506    1.335419           0.552946


In [3]:
def create_era5_data():
    """Simulate ERA5 climate data - mostly complete"""
    data = {}
    features = ['temp_2m', 'precip', 'wind_speed', 'humidity', 'pressure', 'radiation']
    
    for feature in features:
        # Create base data
        if feature == 'temp_2m':
            base = np.random.normal(15, 10, (len(basin_ids), len(dates)))
        elif feature == 'precip':
            base = np.random.exponential(2, (len(basin_ids), len(dates)))
        else:
            base = np.random.normal(50, 20, (len(basin_ids), len(dates)))
        
        # Add some seasonal patterns
        day_of_year = np.array([d.dayofyear for d in dates])
        seasonal = 10 * np.sin(2 * np.pi * day_of_year / 365)
        base = base + seasonal[np.newaxis, :]
                
        data[feature] = (['basin_id', 'date'], base)
    
    return xr.Dataset(
        data, 
        coords={'basin_id': basin_ids, 'date': dates}
    )
era5_ds = create_era5_data()
print(f"ERA5 dataset shape: {era5_ds.sizes}")
print(f"ERA5 features: {list(era5_ds.data_vars)}")

ERA5 dataset shape: Frozen({'basin_id': 1000, 'date': 1827})
ERA5 features: ['temp_2m', 'precip', 'wind_speed', 'humidity', 'pressure', 'radiation']


In [4]:
def create_swot_data():
    """Simulate SWOT satellite data - very sparse"""
    features = ['height', 'width', 'slope']
    data = {}
    
    for feature in features:
        # Start with all NaN
        base = np.full((len(basin_ids), len(dates)), np.nan)
        
        # Only 1-2% of data points are available
        # SWOT has irregular revisit patterns
        for i, basin in enumerate(basin_ids):
            # Each basin gets observations on random days (1-2% of days)
            n_obs = np.random.poisson(len(dates) * 0.015)  # ~1.5% of days
            if n_obs > 0:
                obs_indices = np.random.choice(len(dates), min(n_obs, len(dates)), replace=False)
                if feature == 'water_level':
                    base[i, obs_indices] = np.random.normal(100, 20, len(obs_indices))
                elif feature == 'water_extent':
                    base[i, obs_indices] = np.random.lognormal(2, 1, len(obs_indices))
                else:  # flow_velocity
                    base[i, obs_indices] = np.random.gamma(1, 2, len(obs_indices))
        
        data[feature] = (['basin_id', 'date'], base)
    
    return xr.Dataset(
        data,
        coords={'basin_id': basin_ids, 'date': dates}
    )

swot_ds = create_swot_data()
print(f"SWOT dataset shape: {swot_ds.dims}")
print(f"SWOT features: {list(swot_ds.data_vars)}")

SWOT dataset shape: FrozenMappingWarningOnValuesAccess({'basin_id': 1000, 'date': 1827})
SWOT features: ['height', 'width', 'slope']


In [6]:
test_dir = Path('./h5_data_test')
test_dir.mkdir(exist_ok=True)

# Define chunking strategy based on your access patterns
# You typically access ~1 year of data for random basins
chunk_strategy = {
    'basin_id': min(100, N_BASINS//10),  # ~10% of basins per chunk
    'date': min(365, len(dates)//5),     # ~1 year of data per chunk
}

print(f"Chunking strategy: {chunk_strategy}")

# Save static data
static_df.to_hdf(test_dir / 'static_data.h5', key='static', mode='w', complevel=6)

# Save dynamic data with chunking and compression
era5_ds.to_netcdf(
    test_dir / 'era5_data.nc', 
    engine='h5netcdf',
    encoding={var: {'chunksizes': (chunk_strategy['basin_id'], chunk_strategy['date']), 
                    'compression': 'gzip', 'compression_opts': 6}
              for var in era5_ds.data_vars}
)

swot_ds.to_netcdf(
    test_dir / 'swot_data.nc',
    engine='h5netcdf', 
    encoding={var: {'chunksizes': (chunk_strategy['basin_id'], chunk_strategy['date']),
                    'compression': 'gzip', 'compression_opts': 9}  # Higher compression for sparse data
              for var in swot_ds.data_vars}
)

print("Data saved successfully!")

# Check file sizes
for file in test_dir.glob('*.h5'):
    size_mb = file.stat().st_size / 1024 / 1024
    print(f"{file.name}: {size_mb:.2f} MB")
    
for file in test_dir.glob('*.nc'):
    size_mb = file.stat().st_size / 1024 / 1024
    print(f"{file.name}: {size_mb:.2f} MB")

Chunking strategy: {'basin_id': 100, 'date': 365}
Data saved successfully!
static_data.h5: 0.04 MB
era5_data.nc: 79.28 MB
swot_data.nc: 0.98 MB


In [8]:
def test_access_pattern(n_basins_sample=50, year='2022'):
    """Test loading random basins for one year"""
    
    # Random basin selection
    selected_basins = np.random.choice(basin_ids, n_basins_sample, replace=False)
    
    # Date range for one year
    start_date = f'{year}-01-01'
    end_date = f'{year}-12-31'
    
    print(f"Testing access: {n_basins_sample} basins, year {year}")
    
    # Time the loading
    start_time = time.time()
    
    # Load static data for selected basins
    static_subset = pd.read_hdf(test_dir / 'static_data.h5', key='static')
    static_subset = static_subset.loc[selected_basins]
    
    # Load ERA5 data
    era5_loaded = xr.open_dataset(test_dir / 'era5_data.nc', engine='h5netcdf')
    era5_subset = era5_loaded.sel(
        basin_id=selected_basins, 
        date=slice(start_date, end_date)
    )
    
    # Load SWOT data  
    swot_loaded = xr.open_dataset(test_dir / 'swot_data.nc', engine='h5netcdf')
    swot_subset = swot_loaded.sel(
        basin_id=selected_basins,
        date=slice(start_date, end_date)
    )
    
    load_time = time.time() - start_time
    
    print(f"Loading took: {load_time:.3f} seconds")
    print(f"Static data shape: {static_subset.shape}")
    print(f"ERA5 data shape: {dict(era5_subset.sizes)}")
    print(f"SWOT data shape: {dict(swot_subset.sizes)}")
    
    # Clean up
    era5_loaded.close()
    swot_loaded.close()
    
    return static_subset, era5_subset, swot_subset

# Test the access pattern
static_test, era5_test, swot_test = test_access_pattern()

Testing access: 50 basins, year 2022
Loading took: 0.105 seconds
Static data shape: (50, 4)
ERA5 data shape: {'basin_id': 50, 'date': 365}
SWOT data shape: {'basin_id': 50, 'date': 365}


In [9]:
# Test adding new time series data (simulating your preprocessing workflow)
def add_new_basin_timeseries(basin_id, source='era5'):
    """Simulate adding a new time series for a basin after preprocessing"""
    
    print(f"Testing incremental addition for {basin_id} in {source}")
    
    if source == 'era5':
        # Load existing dataset
        ds = xr.open_dataset(test_dir / 'era5_data.nc', engine='h5netcdf')
        
        # Create new time series data for this basin
        new_data = {}
        for var in ds.data_vars:
            if var == 'temp_2m':
                new_values = np.random.normal(20, 5, len(dates))  # Different from original
            else:
                new_values = np.random.normal(100, 30, len(dates))
            new_data[var] = (['date'], new_values)
        
        # Create new dataset for this basin
        new_ds = xr.Dataset(
            new_data,
            coords={'date': dates}
        )
        
        # Update the specific basin (this is where you'd update after preprocessing)
        ds_updated = ds.copy()
        ds_updated.loc[dict(basin_id=basin_id)] = new_ds
        
        ds.close()
        return ds_updated
    
# Test with a specific basin
test_basin = basin_ids[0]
updated_ds = add_new_basin_timeseries(test_basin)

print(f"Original value for {test_basin}, temp_2m on 2020-01-01:")
original = xr.open_dataset(test_dir / 'era5_data.nc', engine='h5netcdf')
print(original.sel(basin_id=test_basin, date='2020-01-01')['temp_2m'].values)

print(f"Updated value for {test_basin}, temp_2m on 2020-01-01:")
print(updated_ds.sel(basin_id=test_basin, date='2020-01-01')['temp_2m'].values)

original.close()

Testing incremental addition for basin_000000 in era5
Original value for basin_000000, temp_2m on 2020-01-01:
9.00437994636127
Updated value for basin_000000, temp_2m on 2020-01-01:
22.0257332687207


In [10]:
import psutil
import gc

def analyze_memory_usage():
    """Analyze memory usage patterns"""
    
    process = psutil.Process()
    initial_memory = process.memory_info().rss / 1024 / 1024  # MB
    
    print(f"Initial memory usage: {initial_memory:.1f} MB")
    
    # Test lazy loading
    print("\n=== Testing Lazy Loading ===")
    era5_lazy = xr.open_dataset(test_dir / 'era5_data.nc', engine='h5netcdf')
    memory_after_open = process.memory_info().rss / 1024 / 1024
    print(f"Memory after opening (lazy): {memory_after_open:.1f} MB (+{memory_after_open-initial_memory:.1f})")
    
    # Load a subset into memory
    subset = era5_lazy.sel(basin_id=basin_ids[:100], date=slice('2022-01-01', '2022-12-31')).load()
    memory_after_load = process.memory_info().rss / 1024 / 1024
    print(f"Memory after loading subset: {memory_after_load:.1f} MB (+{memory_after_load-memory_after_open:.1f})")
    
    era5_lazy.close()
    del subset
    gc.collect()
    
    # Test different chunk sizes impact
    print(f"\n=== Chunk Size Analysis ===")
    with xr.open_dataset(test_dir / 'era5_data.nc', engine='h5netcdf') as ds:
        for var in list(ds.data_vars)[:2]:  # Test first 2 variables
            chunks = ds[var].chunks
            print(f"{var} chunks: {chunks}")
            chunk_sizes = [np.prod(chunk_dim) for chunk_dim in zip(*chunks)]
            print(f"  Individual chunk sizes: {chunk_sizes[:5]}... (elements)")
            print(f"  Estimated chunk memory: {np.mean(chunk_sizes) * 8 / 1024 / 1024:.2f} MB per chunk")

analyze_memory_usage()

Initial memory usage: 445.9 MB

=== Testing Lazy Loading ===
Memory after opening (lazy): 446.1 MB (+0.2)
Memory after loading subset: 446.1 MB (+0.0)

=== Chunk Size Analysis ===
temp_2m chunks: None


TypeError: zip() argument after * must be an iterable, not NoneType